## Loggers
Logging library gives you some options to log the output called levels. The top level is DEBUG which log everything and is recommended for Development. However, for production INFO level is recommended. You can defin level in three different ways either use the string (level="DEBUG") or use its numeric value (level=10) or use the constant (level=logging.DEBUG). The CRITICAL level gets the highest numerical value as it is tighter, more strict and almost log nothing except critical things, in other words, CRITICAL represents the highest severity, creating a strict filter that silences the noise and only lets catastrophic events through.

We need to use stream in logging because by default, if you don't use stream, Python sends all logs to a place called Standard Error (sys.stderr), when you set stream=sys.stdout, you are telling Python: "Send my logs to Standard Output (sys.stdout) instead." In other words, stream=sys.stdout is a way of protecting your logs from being misunderstood by the outside world. It forces Python to send them down the "normal data" pipe so external systems don't panic.

## GPS Time
GPS time is a continuous integer count of seconds since January 6, 1980 00:00:00 UTC, with no leap seconds - It never pauses or reset.

## MiB VS. MB
MB (Megabyte) uses powers of 10, 1 MB = 1,000,000 bytes, which is used in marketing, storage manufacturers, internet speeds, on the other hand, MiB(Mebibyte) uses powers of 2, 1 MiB = 2^20 = 1,048,576 bytes, which is used more in operating systems, memory calculations, and lowlevel computing. And we are using it because computers are binary and memory naturally aligns with 2^10=1024, 2^20 and 2^30 etc. In LIGO, because numbers are float64, means they are using 64 bits, which are 8 bytes per sample, so if we multiply total_samples * 8 bytes and divide it by 1000000 we will get almost 134 MB and by dividing by 1,048,576, we will get 128MiB which both are correct.

## Spark
In LIGO, let's say for 4096 sec file and at sample rate 4096 HZ, means each second we have 4096 samples, total samples then is 4096 * 4096. It takes 134MB or 128MiB memory.

Now we have two options to store raw data, first option will be store the whole strain into 1 row along with duration, start time, or other information. This is not recommended for spark to have a single row of 134 MB of data. The other option is window the signal (let's say in 2 sec windows) and store them.

### Why is one 134 MB row bad in Spark?
Spark is not only about total data size. It is about how data is divided. Spark breaks a DataFrame/RDD into partitions. For each stage, Spark usually creates one task per partition. A task runs on one executor core. Executors can run multiple tasks in parallel depending on how many cores they have. So Spark parallelism mainly comes from having enough partitions/tasks. A simplified view is 1 partition usually becomes one task, 1 task runs on 1 CPU core, and executors run many tasks in parallel. So parallelism comes from having many partitions/tasks. Also a core might runs many tasks

The bad design is having one giant row. If we store the whole signal as one row, then logically we will end up with one giant object. Even if the dataframe exists, the useful work on that signal is not naturally split across many tasks. For example, if we apply a UDF, Spark sees one row, one function call, one task/core does the heavy work. So our cluster may have many cores, but only one core is really doing that signal's work, which is bad for Spark.

Spark likes many medium/small records and it dislikes one giant record.

### Spark Task
A task is basically a unit of work on one partition (Partition + Operation). So it is a small executable work on one partition. 